# Global XAI For Already-Trained Best Neural Model

This Colab notebook does **not train anything**. It loads the already trained best neural checkpoint, computes global XAI across all 23 MM-IMDb genre labels, and saves every artifact needed by the local dashboard's **Global XAI - Best Neural** tab.

Required files in the project checkout or mounted Drive:

- `dataset/data/multimodal_imdb.hdf5`
- `dataset/data/metadata.npy`
- `outputs/splits/test_indices.npy`
- `outputs/models/best/neural_multimodal_best.pt`

## 1. Mount Drive And Open Project

Edit `COLAB_PROJECT_DIR` only if your repository lives somewhere else on Google Drive.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
COLAB_PROJECT_DIR = Path('/content/drive/MyDrive/Explaining-Predictions-of-Machine-Learning-Models')
GITHUB_REPO = 'https://github.com/SamuelSulan/Explaining-Predictions-of-Machine-Learning-Models.git'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not COLAB_PROJECT_DIR.exists():
        COLAB_PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', GITHUB_REPO, str(COLAB_PROJECT_DIR)], check=True)
    os.chdir(COLAB_PROJECT_DIR)
else:
    os.chdir(Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd())

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project root:', PROJECT_ROOT)

## 2. Install Missing Runtime Dependencies

This keeps Colab's CUDA-enabled PyTorch installation intact and only installs missing extras.

In [ ]:
import importlib.util

missing = []
for module, package in [
    ('captum', 'captum==0.9.0'),
    ('iterstrat', 'iterative-stratification==0.1.9'),
]:
    if importlib.util.find_spec(module) is None:
        missing.append(package)

if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)

## 3. Configure Global XAI Run

`RUN_MODE = 'full'` is intended for final thesis artifacts. Change it to `'smoke'` for a quick check that paths and model loading work.

In [ ]:
RUN_MODE = 'full'  # 'full' for final artifacts, 'smoke' for a quick check

CONFIG_PATH = 'configs/default.yaml'
CHECKPOINT_PATH = 'outputs/models/best/neural_multimodal_best.pt'
OUTPUT_DIR = 'outputs/global_xai/best_neural'
SPLIT = 'test'
SEED = 42

if RUN_MODE == 'smoke':
    SAMPLES_PER_LABEL = 1
    BATCH_SIZE = 4
    TOKEN_CANDIDATES_PER_LABEL = 5
    TOKEN_OCCLUSION_TOP_K = 2
    IMAGE_OCCLUSION_GRID = 2
else:
    SAMPLES_PER_LABEL = 25
    BATCH_SIZE = 16
    TOKEN_CANDIDATES_PER_LABEL = 20
    TOKEN_OCCLUSION_TOP_K = 10
    IMAGE_OCCLUSION_GRID = 4

ENABLE_TOKEN_OCCLUSION = True
ENABLE_IMAGE_OCCLUSION = True

# Optional: if local per-instance XAI exists, the notebook also aggregates saved Layer IG token attributions.
LOCAL_XAI_DIR = 'outputs/colab_test_xai_all_models/run_newest_balanced/xai/best_neural'

print({
    'RUN_MODE': RUN_MODE,
    'CHECKPOINT_PATH': CHECKPOINT_PATH,
    'OUTPUT_DIR': OUTPUT_DIR,
    'SAMPLES_PER_LABEL': SAMPLES_PER_LABEL,
    'TOKEN_OCCLUSION_TOP_K': TOKEN_OCCLUSION_TOP_K,
    'IMAGE_OCCLUSION_GRID': IMAGE_OCCLUSION_GRID,
})

## 4. Verify Required Files

This cell fails early if the already-trained checkpoint or dataset files are missing. There is intentionally no training fallback.

In [ ]:
from mmimdb.data import DatasetPaths
from mmimdb.utils import load_config, resolve_path

config = load_config(CONFIG_PATH)
paths = DatasetPaths.from_config(config)
required_paths = {
    'HDF5 dataset': paths.hdf5,
    'metadata': paths.metadata,
    'train split': resolve_path(config['splits']['output_dir']) / 'train_indices.npy',
    'validation split': resolve_path(config['splits']['output_dir']) / 'val_indices.npy',
    'test split': resolve_path(config['splits']['output_dir']) / 'test_indices.npy',
    'trained best neural checkpoint': resolve_path(CHECKPOINT_PATH),
}

missing_paths = {name: path for name, path in required_paths.items() if not Path(path).exists()}
if missing_paths:
    for name, path in missing_paths.items():
        print(f'MISSING {name}: {path}')
    raise FileNotFoundError('Required files are missing. Upload/copy them to Drive first; this notebook does not train a replacement model.')

for name, path in required_paths.items():
    print(f'OK {name}: {path}')

## 5. Methods Saved For Thesis

The run writes `global_xai_methods_for_thesis.csv` with concise descriptions you can reuse in the thesis.

In [ ]:
import pandas as pd
from mmimdb.global_xai import METHOD_DESCRIPTIONS

pd.DataFrame(METHOD_DESCRIPTIONS)

## 6. Run Global XAI

This loads the trained model from `CHECKPOINT_PATH`, computes all configured global XAI methods, and saves tables/figures/JSON under `OUTPUT_DIR`.

In [ ]:
from mmimdb.global_xai import run_global_neural_xai

local_xai_path = Path(LOCAL_XAI_DIR)
summary = run_global_neural_xai(
    config_path=CONFIG_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    output_dir=OUTPUT_DIR,
    split=SPLIT,
    samples_per_label=SAMPLES_PER_LABEL,
    seed=SEED,
    batch_size=BATCH_SIZE,
    token_candidates_per_label=TOKEN_CANDIDATES_PER_LABEL,
    token_occlusion_top_k=TOKEN_OCCLUSION_TOP_K,
    image_occlusion_grid=IMAGE_OCCLUSION_GRID,
    enable_token_occlusion=ENABLE_TOKEN_OCCLUSION,
    enable_image_occlusion=ENABLE_IMAGE_OCCLUSION,
    local_xai_dir=str(local_xai_path) if local_xai_path.exists() else None,
)

print('Global XAI summary:', Path(OUTPUT_DIR, 'global_xai_summary.json').resolve())
print('Selected samples:', summary['selected_sample_count'])
print('Runtime seconds:', round(summary['runtime_seconds'], 2))
print('Warnings:', len(summary['warnings']))

## 7. Inspect Saved Data

In [ ]:
from IPython.display import Image, display

out = Path(OUTPUT_DIR)
expected_outputs = [
    'global_xai_summary.json',
    'global_xai_methods_for_thesis.csv',
    'prediction_context.csv',
    'modality_ablation_and_permutation.csv',
    'global_modality_shapley.csv',
    'global_token_occlusion.csv',
    'aggregated_lig_tokens.csv',
    'per_label_image_occlusion_heatmaps.csv',
]

for name in expected_outputs:
    path = out / name
    print(('OK   ' if path.exists() else 'MISS ') + str(path))

display(pd.read_csv(out / 'prediction_context.csv').head(23))
display(pd.read_csv(out / 'modality_ablation_and_permutation.csv'))

for figure_name in [
    'global_modality_utilization.png',
    'per_label_modality_utilization.png',
    'global_token_occlusion_top.png',
    'prediction_context.png',
]:
    path = out / figure_name
    if path.exists():
        display(Image(filename=str(path)))

## 8. Package Artifacts

The dashboard can read the folder directly. This zip is convenient if you want to download or copy the global XAI outputs elsewhere.

In [ ]:
import shutil

zip_base = Path(OUTPUT_DIR).with_name(Path(OUTPUT_DIR).name + '_artifacts')
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=Path(OUTPUT_DIR))
print('Packaged artifacts:', zip_path)

if IN_COLAB:
    print('The zip is saved in Drive. You can also download it from the Colab file browser if needed.')